Report-ready aggregate.

In [2]:
%run spark_common.ipynb

D:/Learn Apache Kafka/jars/commons-pool2-2.13.1.jar,D:/Learn Apache Kafka/jars/kafka-clients-3.9.2.jar,D:/Learn Apache Kafka/jars/spark-sql-kafka-0-10_2.13-4.3.0.jar,D:/Learn Apache Kafka/jars/spark-token-provider-kafka-0-10_2.13-4.3.0.jar D:/Learn Apache Kafka/jars/commons-pool2-2.13.1.jar;D:/Learn Apache Kafka/jars/kafka-clients-3.9.2.jar;D:/Learn Apache Kafka/jars/spark-sql-kafka-0-10_2.13-4.3.0.jar;D:/Learn Apache Kafka/jars/spark-token-provider-kafka-0-10_2.13-4.3.0.jar
4.3.0


In [3]:
spark = get_spark("report")

In [4]:
from pyspark.sql import functions as F

In [ ]:
orders = spark.read.parquet("lake/orders")
unique = orders.dropDuplicates(["order_id"])
print(f"rows in the table: {orders.count()}, unique orders: {unique.count()}")
daily = (unique.groupBy("order_date", "city")
         .agg(F.count("*").alias("orders"), F.round(F.sum("revenue"), 2).alias("revenue"))
         .orderBy("order_date", F.desc("revenue"))
         )
daily.write.mode("overwrite").parquet("lake/gold/city_daily_revenue")
daily.coalesce(1).write.mode("overwrite").option("header", True).csv("reports/city_daily_revenue")
daily.show(20)

In [ ]:
# Run the stream job for every 90 seconds
# > papermill stream_job.ipynb output_stream_job.ipynb -p seconds 90

In [ ]:
# Run the shop stream in another terminal
# > papermill shop_stream.ipynb output_shop_stream.ipynb -p first 2001 -p rate 10 -p seconds 40

In [ ]:
# Run this report.ipynb
# > papermill report.ipynb output_report.ipynb